In [1]:
from pysha256 import sha256, padding
from urllib.parse import quote

sprinklers_token = "499eb4b396f3c5b2b66a14fd0a22ac107605f737132ba7c2855291d68bebaa6c"
sprinklers_data = b"command=SprinklersPowerOn"

clock_token = "15a80ed015539ecc9b854e161f503870021dfc089583ca6868992c5098259c48"
clock_data = b"&command=ClockPowerOff&command=NoOp&command=ClockPowerOn"

# the secret is exactly 8 bytes
original_length_sprinklers = 8 + len(sprinklers_data)
original_length_clock = 8 + len(clock_data)

# sha256 padding originally added
padding_sprinklers = padding(original_length_sprinklers)
padding_clock = padding(original_length_clock)

# continue sha256 from the internal state represented by the token
# count = original_length + len(padding) tells pysha256 to prevent to have already processed the secret, the data, and padding 
h_sprinklers = sha256(state=bytes.fromhex(sprinklers_token), count=original_length_sprinklers + len(padding_sprinklers))
h_clock = sha256(state=bytes.fromhex(clock_token), count=original_length_clock + len(padding_clock))

# add the command we want to append
suffix_sprinklers = b"&command=SprinklersPowerOn"
suffix_clock = b"&command=ClockPowerOff&command=NoOp&command=ClockPowerOn"

h_sprinklers.update(suffix_sprinklers)
h_clock.update(suffix_clock)

new_token_sprinklers = h_sprinklers.hexdigest()
new_token_clock = h_clock.hexdigest()

print(f"Sprinklers: {new_token_sprinklers} \n\t {quote(padding_sprinklers + suffix_sprinklers, safe='&=')}")
print(f"Clock: {new_token_clock} \n\t {quote(padding_clock + suffix_clock, safe='&=')}")

Sprinklers: 6df05c222d23bf591bbc67ff47244b5606654435b43106b6fa3fdaba4eaf4147 
	 %80%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%01%08&command=SprinklersPowerOn
Clock: f546957122e5449805d98522403281a29212b57a8da76d3786fd3d7e8713cc8b 
	 %80%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%00%02%00&command=ClockPowerOff&command=NoOp&command=ClockPowerOn
